### Load Data

In [53]:
#Load data
import pandas as pd
import numpy as np

liver_data = pd.read_csv('../NCDB_2022/liver.csv')
liver_data.shape

/var/folders/4r/chd7cdjn77j6krd2g7fstr3h0000gn/T/ipykernel_42904/3296579314.py:5: DtypeWarning: Columns (164,170,173,176,177,179,218,219,220,319,320,322,323,331,332,333,334) have mixed types. Specify dtype option on import or set low_memory=False.
  liver_data = pd.read_csv('../NCDB_2022/liver.csv')


(317111, 340)

### Include HCC

In [54]:
#removed 8171 and 8180 per doctor's suggestion
valid_hist_codes = [8170, 8172, 8173, 8174, 8175]

df = liver_data[liver_data['HISTOLOGY'].isin(valid_hist_codes)]
df['HISTOLOGY'].value_counts()
df['PRIMARY_SITE'].value_counts()

PRIMARY_SITE
C220    284109
Name: count, dtype: int64

In [55]:
df.shape

(284109, 340)

### Include TACE and Systemic

In [56]:
df = df[
    df['DX_CHEMO_STARTED_DAYS'].notna() &
    df['DX_IMMUNO_STARTED_DAYS'].notna() &
    df['RX_SUMM_CHEMO'].isin([1, 2, 3])
]

df.shape

(2418, 340)

### Exclude missing survival data

In [57]:
df = df[
    (df['PUF_VITAL_STATUS'].isin([0, 1])) &
    (df['DX_LASTCONTACT_DEATH_MONTHS'].notna()) &
    (df['DX_LASTCONTACT_DEATH_MONTHS'] != 9999.0)
]

df.shape

(2030, 340)

### Define time and event

In [58]:
df['time'] = df['DX_LASTCONTACT_DEATH_MONTHS']
df['event'] = df['PUF_VITAL_STATUS'].apply(lambda x: 1 if x == 0 else 0)  # 1 = dead = event

### Combine stage variables and exclude invalid stage data

In [59]:
df['ANALYTIC_STAGE_GROUP'].value_counts(), df['ANALYTIC_STAGE_GROUP'].isna().sum(), df['ANALYTIC_STAGE_GROUP'].dtype

(ANALYTIC_STAGE_GROUP
 4    752
 3    723
 2    265
 1    181
 9    108
 8      1
 Name: count, dtype: int64,
 np.int64(0),
 dtype('int64'))

In [60]:
df['COMBINED_STAGE_GROUP'] = df['ANALYTIC_STAGE_GROUP']
df['COMBINED_STAGE_GROUP'] = df['COMBINED_STAGE_GROUP'].where(df['COMBINED_STAGE_GROUP'].isin([1,2,3,4]), np.nan)
df.dropna(subset=['COMBINED_STAGE_GROUP'], inplace=True)
df['COMBINED_STAGE_GROUP'].value_counts(), df['COMBINED_STAGE_GROUP'].isna().sum(), df['COMBINED_STAGE_GROUP'].dtype

(COMBINED_STAGE_GROUP
 4.0    752
 3.0    723
 2.0    265
 1.0    181
 Name: count, dtype: int64,
 np.int64(0),
 dtype('float64'))

In [61]:
df['COMBINED_STAGE_GROUP'] = df['COMBINED_STAGE_GROUP'].astype(int)
df.shape

(1921, 343)

### Filter Year of Diagnosis

In [62]:
df['YEAR_OF_DIAGNOSIS'].describe()

count    1921.000000
mean     2018.127538
std         3.446296
min      2004.000000
25%      2018.000000
50%      2019.000000
75%      2020.000000
max      2021.000000
Name: YEAR_OF_DIAGNOSIS, dtype: float64

In [63]:
df = df[df['YEAR_OF_DIAGNOSIS'] >= 2007]
df.shape

(1858, 343)

### Define sequence label

In [64]:
# df['sequence_group'] = df.apply(
#     lambda row: 0 if row['DX_CHEMO_STARTED_DAYS'] < row['DX_IMMUNO_STARTED_DAYS']
#     else 1,
#     axis=1
# )

# same_day = (df['DX_CHEMO_STARTED_DAYS'] - df['DX_IMMUNO_STARTED_DAYS']).abs() < 1e-6
# df.loc[same_day, 'sequence_group'] = 2

# df['sequence_group'].value_counts()

In [65]:
df['treatment_spacing_tace_ref'] = df['DX_CHEMO_STARTED_DAYS'] - df['DX_IMMUNO_STARTED_DAYS']
df['sequence_group'] = df['treatment_spacing_tace_ref'].apply(
    lambda x: 0 if x < -3 else (1 if x > 3 else 2)
)
df['sequence_group'].value_counts()

sequence_group
0    1212
2     362
1     284
Name: count, dtype: int64

### Define surgery

In [66]:
df['Surgery'] = (~df['RX_SUMM_SURG_PRIM_SITE'].isin([0, 99])).astype(int).map({0: 'No', 1: 'Yes'})
df['Surgery'].value_counts()

Surgery
No     1668
Yes     190
Name: count, dtype: int64

In [67]:
df['DX_DEFSURG_STARTED_DAYS'].describe()

count     189.000000
mean      165.465608
std       172.740012
min         0.000000
25%        44.000000
50%       119.000000
75%       216.000000
max      1141.000000
Name: DX_DEFSURG_STARTED_DAYS, dtype: float64

In [68]:
df['Prior_Surgery'] = (df['DX_DEFSURG_STARTED_DAYS'] < df['DX_CHEMO_STARTED_DAYS']) & (df['DX_DEFSURG_STARTED_DAYS'] < df['DX_IMMUNO_STARTED_DAYS'])
df['Prior_Surgery'] = df['Prior_Surgery'].astype(int).map({0: 'No', 1: 'Yes'})
df['Prior_Surgery'].value_counts()

Prior_Surgery
No     1804
Yes      54
Name: count, dtype: int64

In [69]:
df['Subsequent_Surgery'] = (df['DX_DEFSURG_STARTED_DAYS'] > df['DX_CHEMO_STARTED_DAYS']) & (df['DX_DEFSURG_STARTED_DAYS'] > df['DX_IMMUNO_STARTED_DAYS'])
df['Subsequent_Surgery'] = df['Subsequent_Surgery'].astype(int).map({0: 'No', 1: 'Yes'})
df['Subsequent_Surgery'].value_counts()

Subsequent_Surgery
No     1773
Yes      85
Name: count, dtype: int64

In [70]:
df['Surgery_Between'] = ((df['DX_DEFSURG_STARTED_DAYS'] >= df['DX_CHEMO_STARTED_DAYS']) & (df['DX_DEFSURG_STARTED_DAYS'] <= df['DX_IMMUNO_STARTED_DAYS'])) | \
                        ((df['DX_DEFSURG_STARTED_DAYS'] >= df['DX_IMMUNO_STARTED_DAYS']) & (df['DX_DEFSURG_STARTED_DAYS'] <= df['DX_CHEMO_STARTED_DAYS']))
df['Surgery_Between'] = df['Surgery_Between'].astype(int).map({0: 'No', 1: 'Yes'})
df['Surgery_Between'].value_counts()

Surgery_Between
No     1808
Yes      50
Name: count, dtype: int64

### Recode other covariates

In [71]:
covariates = [
    'AGE', 'SEX', 'RACE', 'SPANISH_HISPANIC_ORIGIN', #Demographics
    'INSURANCE_STATUS', 'MED_INC_QUAR_2020', 'NO_HSD_QUAR_2020',#Socioeconomic
    'CDCC_TOTAL_BEST', 'HISTOLOGY', 'TUMOR_SIZE_SUMMARY_2016', 'COMBINED_STAGE_GROUP', 'COMBINED_CLIN_T', #Clinical
    'FACILITY_TYPE_CD', 'FACILITY_LOCATION_CD', #Facility
    'YEAR_OF_DIAGNOSIS' #Temporal
]

#Recode Clinical T
import re

df['COMBINED_CLIN_T'] = df['AJCC_TNM_CLIN_T'].combine_first(df['TNM_CLIN_T'])
df['COMBINED_CLIN_T'] = df['COMBINED_CLIN_T'].astype(str).apply(
    lambda x: re.search(r'\d+', x).group() if re.search(r'\d+', x) else np.nan
)

#Recode year of diagnosis
def collapse_year(year):
    if year >= 2020:
        return '2020+'
    elif year >= 2017:
        return '2017–2019'
    else:
        return '≤2016'

df['YEAR_OF_DIAGNOSIS'] = df['YEAR_OF_DIAGNOSIS'].apply(collapse_year)


# Recode SEX
df['SEX'] = df['SEX'].replace({1: 'Male', 2: 'Female'})

# Recode RACE
df['RACE'] = df['RACE'].astype(str).map({
    '1': 'White',
    '2':'Black',
    '96':'Asian',
    '4':'Asian',
    '98':'Other',
    '10':'Asian',
    '6':'Asian',
    '99':np.nan,
    '3':'Other',
    '8':'Asian',
    '15':'Asian',
    '16':'Asian',
    '5':'Asian',
    '7':'Other',
    '11':'Asian',
    '97':'Other',
    '14':'Asian',
    '17':'Asian',
    '27':'Other',
    '13':'Asian'
}).fillna('Other')

# Recode INSURANCE_STATUS
df['INSURANCE_STATUS'] = df['INSURANCE_STATUS'].map({
    0: 'No insurance',
    1: 'Private/Managed Care',
    2: 'Medicaid',
    3: 'Medicare',
    4: 'Other Government',
    9: 'Unknown'
})

# Recode area_MedianIncome2016_2020 from MED_INC_QUAR_2020
df['MEDIAN_INCOME'] = df['MED_INC_QUAR_2020'].map({
    1: '<$46,277',
    2: '$46,277-$57,856',
    3: '$57,857-$74,062',
    4: '>$74,063'
})

# Recode Charlson-Deyo Score
df['Charlson_Deyo_Score'] = df['CDCC_TOTAL_BEST'].map({
    0: '0',
    1: '1',
    2: '2',
    3: '3'
})

# Recode area_NoHighSchooldegree2016_2020
df['NoHighSchooldegree2016_2020'] = df['NO_HSD_QUAR_2020'].map({
    1: '>=15.3%',
    2: '9.1%-15.2%',
    3: '5.0%-9.0%',
    4: '<5.0%'
})

# Recode facility type
df['FACILITY_TYPE'] = df['FACILITY_TYPE_CD'].map({
    1: 'Community Cancer Program',
    2: 'Comprehensive Community Cancer Program',
    3: 'Academic/Research Program',
    4: 'Integrated Network Cancer Program'
})

# Recode HISTOLOGY for HCC
hcc_hist_map = {
    8170: 'HCC, NOS',
    8171: 'Fibrolamellar HCC',
    8172: 'Scirrhous HCC',
    8173: 'Spindle cell HCC',
    8174: 'Clear cell HCC',
    8175: 'Pleomorphic HCC',
    8180: 'Mixed HCC-Cholangiocarcinoma'
}
df['HISTOLOGY'] = df['HISTOLOGY'].map(hcc_hist_map)

# Recode the region values using map()
region_map = {
    1: "New England",
    2: "Middle Atlantic",
    3: "South Atlantic",
    4: "East North Central",
    5: "East South Central",
    6: "West North Central",
    7: "West South Central",
    8: "Mountain",
    9: "Pacific"
}

df['FACILITY_REGION'] = df['FACILITY_LOCATION_CD'].map(region_map)

#Recode hispanic origin
def recode_hispanic_origin(x):
    if x == 0:
        return "Non-Hispanic"
    elif x in [1, 2, 3, 4, 5, 6, 8]:
        return "Hispanic"
    elif x == 7:
        return "Likely Hispanic"
    else:
        return np.nan

df['HISPANIC'] = df['SPANISH_HISPANIC_ORIGIN'].apply(recode_hispanic_origin)

# Recode tumor size
df['TUMOR_SIZE_SUMMARY_2016'] = pd.to_numeric(df['TUMOR_SIZE_SUMMARY_2016'], errors='coerce')
df.loc[df['TUMOR_SIZE_SUMMARY_2016'] > 300, 'TUMOR_SIZE_SUMMARY_2016'] = np.nan

def bin_tumor_size(x):
    if pd.isna(x):
        return np.nan
    elif x == 0:
        return 'No tumor'
    elif x <= 30:
        return '≤3 cm'
    elif x <= 50:
        return '3–5 cm'
    elif x <= 100:
        return '5–10 cm'
    elif x <= 200:
        return '10–20 cm'
    else:
        return '>20 cm'
    
df['TUMOR_SIZE'] = df['TUMOR_SIZE_SUMMARY_2016'].apply(bin_tumor_size)

df['Radiation_Therapy'] = df['PHASE_I_RT_MODALITY'].apply(
    lambda x: 'No' if x == 0 else ('Unknown' if x == 99 else 'Yes')
)

In [72]:
#Updated covariates list with new names
covariates = [
    'AGE', 'SEX', 'RACE', 'HISPANIC', #Demographics
    'INSURANCE_STATUS', 'MEDIAN_INCOME', 'NoHighSchooldegree2016_2020',#Socioeconomic
    'Charlson_Deyo_Score', 'HISTOLOGY', 'TUMOR_SIZE', 'COMBINED_STAGE_GROUP', 'COMBINED_CLIN_T', 'Radiation_Therapy', #Clinical
    'FACILITY_TYPE', 'FACILITY_REGION', #Facility
    'YEAR_OF_DIAGNOSIS', #Temporal
    'Subsequent_Surgery'  # Surgery status
]

In [73]:
df[covariates].isna().sum()

AGE                              0
SEX                              0
RACE                             0
HISPANIC                        29
INSURANCE_STATUS                 0
MEDIAN_INCOME                  357
NoHighSchooldegree2016_2020    354
Charlson_Deyo_Score              0
HISTOLOGY                        0
TUMOR_SIZE                     400
COMBINED_STAGE_GROUP             0
COMBINED_CLIN_T                 94
Radiation_Therapy                0
FACILITY_TYPE                   45
FACILITY_REGION                 45
YEAR_OF_DIAGNOSIS                0
Subsequent_Surgery               0
dtype: int64

### Final covariates list

In [74]:
#Final covariates list
covariates = [
    'AGE', 'SEX', 'RACE',
    'INSURANCE_STATUS',
    'Charlson_Deyo_Score', 'HISTOLOGY', 'COMBINED_STAGE_GROUP',
    'YEAR_OF_DIAGNOSIS', 'Surgery', 'Radiation_Therapy'
]
print(df['INSURANCE_STATUS'].isna().sum())
print(df['Radiation_Therapy'].isna().sum())

df = df.dropna(subset=covariates)
df['sequence_group'].value_counts()

0
0


sequence_group
0    1212
2     362
1     284
Name: count, dtype: int64

In [75]:
df.shape

(1858, 357)

In [76]:
df.to_csv('analysis_ready.csv', index=False)

In [77]:
df_test = df.copy()
df_test = df_test[df_test['COMBINED_STAGE_GROUP'].isin([3, 4])]
df_test.shape

(1426, 357)

In [78]:
df_test['Radiation_Therapy'].value_counts()

Radiation_Therapy
No         1016
Yes         394
Unknown      16
Name: count, dtype: int64